In [2]:
:dep smartcore = { version = "0.5.0", features = ["ndarray-bindings", "serde"] }
:dep csv = "1.1"
:dep ndarray = { version = "0.16", features = ["rayon"] }
:dep linfa = { version = "0.8.1" }
:dep rand = "0.8"
:dep rand_chacha = "0.3"
:dep plotters = "0.3"
:dep linfa-preprocessing =  "0.8.1" 
:dep linfa-elasticnet = { version =  "0.8.1" }
:dep linfa-reduction =  "0.8.1" 
:dep polars = { version = "0.53", features = ["ndarray"] }
:dep polars-core = "0.53"
:dep image = "0.24"
:dep linfa-clustering="0.8.1"
:dep linfa-nn="0.8.1"
:opt 3

Optimization: 3


In [27]:
use smartcore::{neighbors::knn_classifier::*,linalg::basic::matrix::DenseMatrix,
                naive_bayes::gaussian::GaussianNB,tree::decision_tree_classifier::DecisionTreeClassifier,
                metrics::distance::Distances,tree::decision_tree_classifier::DecisionTreeClassifierParameters,
                tree::decision_tree_classifier::SplitCriterion,ensemble::random_forest_classifier};
use csv;
use linfa::{Dataset};
use ndarray::{Array1, Array2, ArrayView1, ArrayView2, Axis, Ix1, array,s};
use rand::SeedableRng;
use rand_chacha::ChaCha12Rng;
use linfa::prelude::*;
use linfa_reduction::Pca;
use plotters::prelude::*;
use ndarray::Array2;
use std::path::Path as StdPath;
use linfa_preprocessing::linear_scaling::LinearScaler;
use plotters::prelude::*;
use std::path::Path as StdPath;
use polars_core::prelude::*;
use ndarray::Dim;
use ndarray::ArrayBase;
use ndarray::OwnedRepr;
use polars::prelude::CsvReadOptions;
use polars::prelude::SerReader;
use linfa_clustering::{Dbscan,KMeans,Optics};
use rand::SeedableRng;
use rand_chacha::ChaCha8Rng;
use linfa_nn::distance::L2Dist;
use smartcore::xgboost::*;

In [16]:
let mut df = CsvReadOptions::default()
        .with_has_header(false)
        .try_into_reader_with_file_path(Some("DroneTelemetry.csv".into())).unwrap()
        .finish().unwrap();

In [17]:
println!("{:?}",df)

shape: (554, 22)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ column_1 ┆ column_2  ┆ column_3  ┆ column_4  ┆ … ┆ column_19 ┆ column_20 ┆ column_21 ┆ column_22 │
│ ---      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str      ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Drone ID ┆ Applicati ┆ Drone     ┆ Drone     ┆ … ┆ Obstacles ┆ Flight    ┆ Regulator ┆ Notes     │
│          ┆ on        ┆ Size      ┆ Model     ┆   ┆ Encounter ┆ Status    ┆ y         ┆           │
│          ┆           ┆           ┆           ┆   ┆ ed        ┆           ┆ Approval  ┆           │
│          ┆           ┆           ┆           ┆   ┆           ┆           ┆ ID        ┆           │
│ D001     ┆ Package   ┆ Medium    ┆ FlyHigh   ┆ … ┆ No        ┆ Completed

()

In [18]:
let mut df = CsvReadOptions::default()
    .with_has_header(true)
    .try_into_reader_with_file_path(Some("DroneTelemetry.csv".into())).unwrap()
    .finish().unwrap();
let numeric_cols = [
    "Propeller Count",
    "Max Carry Weight (kg)",
    "Actual Carry Weight (kg)",
    "Altitude (meters)",
    "Flight Duration (minutes)",
    "Distance Flown (km)",
    "Battery Remaining (%)",
    "GPS Accuracy (meters)",
    "Wind Speed (m/s)"
];

let mut numeric_data = Vec::new();
let mut drone_ids = Vec::new();

for row_idx in 0..df.height() {
    let drone_id_val = df.column("Drone ID").unwrap().get(row_idx);
    let drone_id = match drone_id_val {
        Ok(val) => val.to_string(),
        Err(_) => "unknown".to_string()
    };
    
    let mut row_values = Vec::new();
    let mut has_numeric = true;
    
    for col_name in &numeric_cols {
        if let Ok(series) = df.column(col_name) {
            match series.get(row_idx) {
                Ok(val) => {
                    let val_str = val.to_string();
                    if let Ok(num) = val_str.parse::<f64>() {
                        row_values.push(num);
                    } else {
                        has_numeric = false;
                        break;
                    }
                },
                Err(_) => {
                    has_numeric = false;
                    break;
                }
            }
        }
    }
    
    if has_numeric && !row_values.is_empty() {
        numeric_data.push(row_values);
        drone_ids.push(drone_id);
    }
}

let shape = (numeric_data.len(), numeric_data[0].len());
let x_vec: Vec<f64> = numeric_data.into_iter().flatten().collect();
let x_final = ndarray::Array2::from_shape_vec(shape, x_vec).unwrap();
let y_final = ndarray::Array1::from_iter(0..shape.0);
let dataset = Dataset::new(x_final, y_final);

let scaler: LinearScaler<f64> = LinearScaler::standard().fit(&dataset).expect("scale e");
let dataset = scaler.transform(dataset);

let pca: Pca<f64> = Pca::params(2).fit(&dataset).unwrap();
let records = pca.predict(&dataset);
let records_dataset = Dataset::from(records.clone());
let n_clusters = 3;
let kmeans_model:KMeans<f64, L2Dist> = KMeans::params(n_clusters)
    .fit(&records_dataset)
    .expect("kmeans errr");
let kmeans_labels = kmeans_model.predict(&records_dataset);
let root = BitMapBackend::new("drone_kmeans_clusters.png", (4000, 4000)).into_drawing_area();
root.fill(&WHITE).unwrap();

let x_min = records.column(0).iter().fold(f64::INFINITY, |a, &b| a.min(b));
let x_max = records.column(0).iter().fold(f64::NEG_INFINITY, |a, &b| a.max(b));
let y_min = records.column(1).iter().fold(f64::INFINITY, |a, &b| a.min(b));
let y_max = records.column(1).iter().fold(f64::NEG_INFINITY, |a, &b| a.max(b));

let x_padding = (x_max - x_min) * 0.1;
let y_padding = (y_max - y_min) * 0.1;
let x_range = (x_min - x_padding)..(x_max + x_padding);
let y_range = (y_min - y_padding)..(y_max + y_padding);

let mut chart = ChartBuilder::on(&root)
    .caption("KMeans Clusters", ("sans-serif", 40))
    .margin(30)
    .x_label_area_size(40)
    .y_label_area_size(40)
    .build_cartesian_2d(x_range, y_range).unwrap();

let colors = [RED, BLUE, GREEN, CYAN, MAGENTA, YELLOW, RGBColor(255, 165, 0), RGBColor(128, 0, 128)];

for id in 0..drone_ids.len() {
    let row = records.row(id);
    let cluster_id = kmeans_labels[id] as usize;
    let color = colors[cluster_id % colors.len()];
    
    chart.draw_series(std::iter::once(
        Circle::new((row[0], row[1]), 5, color.filled())
    )).unwrap();
    
    chart.draw_series(std::iter::once(
        Text::new(
            drone_ids[id].clone(),
            (row[0] + 0.05, row[1] + 0.05),
            ("sans-serif", 12).into_font()
        )
    )).unwrap();
}

root.present().unwrap();

let img = image::open("drone_kmeans_clusters.png").expect("open err");
let scaled = img.resize(1000, 1000, image::imageops::FilterType::Lanczos3);
scaled.save("drone_kmeans_clusters_scaled.png").expect("save err");

In [103]:
use std::collections::HashMap;
use smartcore::neighbors::knn_regressor::*;
use smartcore::tree::decision_tree_regressor::*;
use smartcore::ensemble::random_forest_regressor::*;
use smartcore::xgboost::{XGRegressor, XGRegressorParameters};

let path = "DroneTelemetry.csv";
let mut reader = csv::Reader::from_path(path).unwrap();
let mut rng = ChaCha12Rng::seed_from_u64(5);
let mut manufacture_map: HashMap<String, usize> = HashMap::new();
let mut all_data: Vec<(Vec<f64>, f64, String)> = Vec::new();

for data in reader.records() {
    let rec = data.unwrap();
    if rec[4].trim() == "Manufacture" {
        continue;
    }
    
    let manufacture = rec[4].to_string();
    let parse_results = vec![
        rec[5].parse::<f64>(),
        rec[6].parse::<f64>(),
        rec[7].parse::<f64>(),
        rec[10].parse::<f64>(),
        rec[11].parse::<f64>(),
        rec[12].parse::<f64>(),
        rec[17].parse::<f64>(),
    ];
    let parse_target = rec[15].parse::<f64>();
    
    if parse_results.iter().all(|r| r.is_ok()) && parse_target.is_ok() {
        let features: Vec<f64> = parse_results.into_iter().map(|r| r.unwrap()).collect();
        let target = parse_target.unwrap();
        
        if !manufacture_map.contains_key(&manufacture) {
            manufacture_map.insert(manufacture.clone(), manufacture_map.len());
        }
        
        all_data.push((features, target, manufacture));
    }
}
let num_samples = all_data.len();
let num_manufacture_categories = manufacture_map.len();
let num_features = 7 + num_manufacture_categories; 
let mut records = Array2::zeros((num_samples, num_features));
let mut targets = Array1::zeros(num_samples);
for (i, (features, target, manufacture)) in all_data.iter().enumerate() {
    for j in 0..7 {
        records[[i, j]] = features[j];
    }
    if let Some(&category_index) = manufacture_map.get(manufacture) {
        records[[i, 7 + category_index]] = 1.0;
    }
    
    targets[i] = *target;
}
let dataset = Dataset::new(records, targets);
let (train_data, test_data) = dataset.shuffle(&mut rng).split_with_ratio(0.7);
let x = DenseMatrix::from_2d_vec(
    &train_data.records().axis_iter(Axis(0))
        .map(|row| row.to_vec())
        .collect::<Vec<Vec<f64>>>()
).unwrap();
let y: Vec<f64> = train_data.targets()
    .map(|value| *value)
    .to_vec();
let x_train = DenseMatrix::from_2d_vec(
    &train_data.records().axis_iter(Axis(0))
        .map(|row| row.to_vec())
        .collect::<Vec<Vec<f64>>>()
).unwrap();
let y_train: Vec<f64> = train_data.targets()
    .map(|value| *value)
    .to_vec();
let test_x = DenseMatrix::from_2d_vec(
    &test_data.records().axis_iter(Axis(0))
        .map(|row| row.to_vec())
        .collect::<Vec<Vec<f64>>>()
).unwrap();
let test_y: Vec<f64> = test_data.targets()
    .map(|value| *value)
    .to_vec();
let knn_params = KNNRegressorParameters::default()
    .with_k(2)
    .with_distance(Distances::manhattan());
let knn = KNNRegressor::fit(&x_train, &y_train, knn_params).unwrap();
let dt_params = DecisionTreeRegressorParameters::default()
    .with_max_depth(25)
    .with_min_samples_leaf(0)
    .with_min_samples_split(0);
let dt = DecisionTreeRegressor::fit(&x_train, &y_train, dt_params).unwrap();
let rf_params = RandomForestRegressorParameters::default();
let rf = RandomForestRegressor::fit(&x_train, &y_train, rf_params).unwrap();
let boost_params =  XGRegressorParameters::default()
    .with_n_estimators(50)
    .with_max_depth(3)
    .with_learning_rate(0.1);
let boost = XGRegressor::fit(&x_train, &y_train, boost_params).unwrap();
let knn_res = knn.predict(&test_x).unwrap();
let dt_res = dt.predict(&test_x).unwrap();
let rf_res = rf.predict(&test_x).unwrap();
let boost_res = boost.predict(&test_x).unwrap();
let mut knn_mae = 0.0;
let mut dt_mae = 0.0;
let mut rf_mae = 0.0;
let mut knn_mse = 0.0;
let mut dt_mse = 0.0;
let mut rf_mse = 0.0;
for i in 0..test_y.len() {
    let knn_error = (test_y[i] - knn_res[i]).abs();
    knn_mae += knn_error;
    knn_mse += knn_error * knn_error;
    let dt_error = (test_y[i] - dt_res[i]).abs();
    dt_mae += dt_error;
    dt_mse += dt_error * dt_error;
    let rf_error = (test_y[i] - rf_res[i]).abs();
    rf_mae += rf_error;
    rf_mse += rf_error * rf_error;
}
let n = test_y.len() as f64;
knn_mae /= n;
dt_mae /= n;
rf_mae /= n;

knn_mse /= n;
dt_mse /= n;
rf_mse /= n;
let knn_rmse = knn_mse.sqrt();
let dt_rmse = dt_mse.sqrt();
let rf_rmse = rf_mse.sqrt();
let y_mean = test_y.iter().sum::<f64>() / n;
let mut ss_tot = 0.0;
let mut ss_res_knn = 0.0;
let mut ss_res_dt = 0.0;
let mut ss_res_rf = 0.0;
let mut ss_res_boost = 0.0;
for i in 0..test_y.len() {
    ss_tot += (test_y[i] - y_mean).powi(2);
    ss_res_knn += (test_y[i] - knn_res[i]).powi(2);
    ss_res_dt += (test_y[i] - dt_res[i]).powi(2);
    ss_res_rf += (test_y[i] - rf_res[i]).powi(2);
    ss_res_boost += (test_y[i] - boost_res[i]).powi(2);
}
let knn_r2 = 1.0 - (ss_res_knn / ss_tot);
let dt_r2 = 1.0 - (ss_res_dt / ss_tot);
let rf_r2 = 1.0 - (ss_res_rf / ss_tot);
let boost_r2 = 1.0 - (ss_res_boost / ss_tot);
println!("KNN:           MAE={:.4}, RMSE={:.4}, R2={:.4}", knn_mae, knn_rmse, knn_r2);
println!("DT: MAE={:.4}, RMSE={:.4}, R2={:.4}", dt_mae, dt_rmse, dt_r2);
println!("RF: MAE={:.4}, RMSE={:.4}, R2={:.4}", rf_mae, rf_rmse, rf_r2);
println!("BOOST: MAE={:.4}, RMSE={:.4}, R2={:.4}", rf_mae, rf_rmse, rf_r2);
let tolerance = 1.0; 
let knn_good = knn_res.iter().zip(test_y.iter())
    .filter(|(pred, actual)| (*pred - *actual).abs() < tolerance)
    .count() as f64 / n * 100.0;
let dt_good = dt_res.iter().zip(test_y.iter())
    .filter(|(pred, actual)| (*pred - *actual).abs() < tolerance)
    .count() as f64 / n * 100.0;
let boost_good = boost_res.iter().zip(test_y.iter())
    .filter(|(pred, actual)| (*pred - *actual).abs() < tolerance)
    .count() as f64 / n * 100.0;
let rf_good = rf_res.iter().zip(test_y.iter())
    .filter(|(pred, actual)| (*pred - *actual).abs() < tolerance)
    .count() as f64 / n * 100.0;
println!("Предсказаний с ошибкой < {:.1}:", tolerance);
println!("KNN: {:.1}%, DT: {:.1}%, RF: {:.1}%,BOOST: {:.1}%", knn_good, dt_good,rf_good,boost_good);

KNN:           MAE=1.1524, RMSE=1.8959, R2=0.9906
DT: MAE=1.0000, RMSE=2.1069, R2=0.9884
RF: MAE=1.5483, RMSE=2.3398, R2=0.9856
BOOST: MAE=1.5483, RMSE=2.3398, R2=0.9856
Предсказаний с ошибкой < 1.0:
KNN: 53.0%, DT: 67.7%, RF: 50.6%,BOOST: 29.9%
